# Initial Model Training

We used 2 different datasets and found that the "spotify_10000_dualgrams.csv" dataset performed better than the other.

Since the training process was conducted on a '.py' file and it took a lot of time to run all codes, we could not include the execution results in this file.

## Imports

In [5]:
%pip install scikit-learn
import pandas as pd
import numpy as np
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from heapq import nlargest
from sklearn.ensemble import StackingClassifier
import traceback

Note: you may need to restart the kernel to use updated packages.


## Defining Helper Functions

In [6]:
# To split train/test data, test_size=0.30
def dataset_train_test_split_stratified_for_sub_type(df):
    try:
        df = df.dropna(how="any", axis=0)
        df.reset_index(drop=True, inplace=True)
        df.playlist_genre = pd.to_numeric(df.playlist_genre)

        if "data_type" not in df.columns:
            X_train, X_test, y_train, y_test = train_test_split(df.loc[:, df.columns != 'playlist_genre'],
                                                                df["playlist_genre"], test_size=0.30, random_state=0)
            df.loc[X_test.index, 'data_type'] = 'test'
            df.data_type = df.data_type.fillna('train')
        else:
            X_train = df[df['data_type'] == 'train']["text_clean"]
            y_train = df[df['data_type'] == 'train']["label"]

            X_test = df[df['data_type'] == 'test']["text_clean"]
            y_test = df[df['data_type'] == 'test']["label"]

        category, count = np.unique(y_train, return_counts=True)

        return X_train, y_train, X_test, y_test
    except Exception as e:
        print(e)

# To generate classification report which contains information that can evaluate a model
def classification_report_generator(actual, predicted, model_name):
    report = classification_report(actual, predicted, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(
        f"{model_name}_classification_report.csv")
    return report

# label dictionary
label_dictionary = {'rock': 0, 'r&b': 1, 'pop': 2, 'edm': 3, 'rap': 4, 'latin': 5}

# To generate confusion matrix
def confusion_matrix_generator(actual, predicted, model_name):
    cbar_kws = {
        'orientation': 'vertical',
        'shrink': 1,
        'extend': 'min',
        'extendfrac': 0.1,
        'drawedges': True
    }
    fig, ax = plt.subplots(2, 1, figsize=(30, 30))
    cm = confusion_matrix(actual, predicted)
    cmn = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    dataplot = sns.heatmap(
        cmn, annot=True, annot_kws={'size': 20}, fmt='.2f', ax=ax[0], cmap=plt.cm.Blues,
        cbar_kws=cbar_kws, square=True
    )
    xticklabels = [tick.get_text() for tick in ax[0].get_xticklabels()]
    yticklabels = [tick.get_text() for tick in ax[0].get_yticklabels()]

    # Replace the tick labels with dictionary values
    xticklabels = [list(label_dictionary.keys())[list(label_dictionary.values()).index(int(label))] for label in
                   xticklabels]
    yticklabels = [list(label_dictionary.keys())[list(label_dictionary.values()).index(int(label))] for label in
                   yticklabels]

    # Set the updated tick labels
    ax[0].set_xticklabels(xticklabels)
    ax[0].set_yticklabels(yticklabels)
    ax[0].set_title("Normalized Confusion matrix", size=20)

    ax[0].set_xlabel("Predicted", size=20)
    ax[0].set_ylabel("Actual", size=20)
    
    plt.setp(dataplot.get_yticklabels(),
             rotation=0,
             ha='right',
             rotation_mode='anchor',
             size=20
             )

    plt.setp(dataplot.get_xticklabels(),
             rotation=45,
             ha='right',
             rotation_mode='anchor',
             size=20
             )

    dataplot = sns.heatmap(
        cm, annot=True, fmt='d', annot_kws={'size': 20}, ax=ax[1], cmap=plt.cm.Blues,
        cbar_kws=cbar_kws, square=True
    )
    xticklabels = [tick.get_text() for tick in ax[1].get_xticklabels()]
    yticklabels = [tick.get_text() for tick in ax[1].get_yticklabels()]

    # Replace the tick labels with dictionary values
    xticklabels = [list(label_dictionary.keys())[list(label_dictionary.values()).index(int(label))] for label in
                   xticklabels]
    yticklabels = [list(label_dictionary.keys())[list(label_dictionary.values()).index(int(label))] for label in
                   yticklabels]

    # Set the updated tick labels
    ax[1].set_xticklabels(xticklabels)
    ax[1].set_yticklabels(yticklabels)
    ax[1].set_title("Confusion matrix", size=20)
    ax[1].set_xlabel("Predicted", size=20)
    ax[1].set_ylabel("Actual", size=20)

    plt.setp(dataplot.get_yticklabels(),
             rotation=0,
             ha='right',
             rotation_mode='anchor',
             size=20
             )

    plt.setp(dataplot.get_xticklabels(),
             rotation=45,
             ha='right',
             rotation_mode='anchor',
             size=20
             )
    plt.savefig(f'{model_name}_confusion_matrix')


macro_f1 = {}

# define classifier trainer to train the model 
# this classifier will save a sav file, generate a classification report and draw a confusion matrix automatically
# also, it will print each trained model's evaluation scores
# avoid interruption when errors occur, since the running time would be extremely long
def classifier_trainer(model, model_name, X_train, y_Train, X_test, y_test):
    try:
        model.fit(X_train, y_Train)
        joblib.dump(model, f"{model_name}_sclf.sav")
        y_predicted = model.predict(X_test)
        report = classification_report_generator(y_test, y_predicted, model_name)
        confusion_matrix_generator(y_test, y_predicted, model_name)

        if model_name != "SC":
            macro_f1[model_name] = float(report['macro avg']['f1-score'])
            print("Model Trained: %s, Report: %s" % (model_name, report))
        else:
            print("Model Trained: %s, Report: %s" % (model_name, report))
            return model
    except Exception as e:
        print("Error: %s, traceback: %s" % (e, traceback.format_exc()))

# define a function that can train 8 models find top 5 models by their macro f1, train Stacking Classifier model and
# generate sav file
# avoid interruption when errors occur, since the running time would be extremely long
def custom_model_initializer(X_train, y_Train, X_test, y_test, model_name):
    try:

        LR = LogisticRegression(solver='lbfgs', max_iter=3000)
        classifier_trainer(LR, "LR", X_train, y_Train, X_test, y_test)

        XGB = XGBClassifier(gamma=0.001, n_estimators=150, max_depth=5, max_bin=10)
        classifier_trainer(XGB, "XGB", X_train, y_Train, X_test, y_test)

        KNN = KNeighborsClassifier(n_neighbors=15)
        classifier_trainer(KNN, "KNN", X_train, y_Train, X_test, y_test)

        DT = tree.DecisionTreeClassifier()
        classifier_trainer(DT, "DT", X_train, y_Train, X_test, y_test)

        RFC = RandomForestClassifier(max_depth=20, min_samples_leaf=1, min_samples_split=5, n_estimators=200)
        classifier_trainer(RFC, "RFC", X_train, y_Train, X_test, y_test)

        SVC = svm.SVC(kernel='rbf', random_state=0, probability=True)
        classifier_trainer(SVC, "SVC", X_train, y_Train, X_test, y_test)

        ABC = AdaBoostClassifier(n_estimators=600, algorithm="SAMME.R", learning_rate=0.1)
        classifier_trainer(ABC, "ABC", X_train, y_Train, X_test, y_test)

        GBC = GradientBoostingClassifier(n_estimators=300, learning_rate=1.0, max_depth=7)
        classifier_trainer(GBC, "GBC", X_train, y_Train, X_test, y_test)

        model_abbrev = {'KNN': ('KNN', KNN),
                        'GBC': ('GBC', GBC),
                        'SVC': ('SVC', SVC),
                        'DT': ('DT', DT),
                        'XGB': ('XGB', XGB),
                        'RFC': ('RFC', RFC),
                        'LR': ('LR', LR),
                        'ABC': ('ABC', ABC)
                        }
        # Checking Macro_F1
        five_highest = nlargest(5, macro_f1, key=macro_f1.get)
        print("Top 5 Classifiers: %s" % five_highest)

        estimators = [model_abbrev[i] for i in five_highest]

        SC = StackingClassifier(estimators=estimators,
                                final_estimator=LogisticRegression(solver='lbfgs', max_iter=3000))
        model = classifier_trainer(SC, "SC", X_train, y_Train, X_test, y_test)

        filename = f'{model_name}_sclf.sav'
        joblib.dump(model, filename)

    except Exception as e:
        print("Error: %s, traceback: %s" % (e, traceback.format_exc()))

## Model Training

In [8]:
# import data
df = pd.read_csv("spotify_10000_dualgrams.csv", index_col=0)

# drop lyrics column
df.drop(columns="lyrics", inplace=True)

# check data
df.head()

,track_popularity_F,loudness_F,tempo_F,duration_ms_F,playlist_genre,danceability_F,energy_F,key_F,mode_F,speechiness_F,...,yuuuuuuu crank,zero,zig,zip,zombi,zombi zombi,zone,zoo,zoom,zoom zoom
0,-0.562509,0.720530,0.531096,2.440463,0,0.303,0.880,9,1,0.0442,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.697676,-0.215446,-0.326873,0.511656,1,0.845,0.652,6,0,0.2160,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.035468,0.354941,-0.080788,0.174686,1,0.425,0.378,5,0,0.0341,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.937532,1.650075,0.339533,-0.692178,2,0.760,0.887,9,1,0.0409,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.140240,0.240525,0.992401,-0.143993,1,0.496,0.639,6,1,0.0550,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# split train / test set
X_train, y_train, X_test, y_test = dataset_train_test_split_stratified_for_sub_type(df)

In [ ]:
# train models, get Stacking Classifier model as "main"
custom_model_initializer(X_train, y_train, X_test, y_test, "main")